# HindiMix-Noisy — CharCNN (medium + high noise only)
**This notebook only runs CharCNN for medium and high noise levels.**
Send the 2 output JSON files back: `char_cnn_medium.json` and `char_cnn_high.json`

Runtime: CPU is fine (~15 min total)

## Step 1: Mount Google Drive (to save results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS_DIR = '/content/drive/MyDrive/hindiMix_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Results will save to:', RESULTS_DIR)

## Step 2: Upload data files
Upload these 4 files from `data/final/`:
- `train.csv`
- `val.csv`
- `test_noisy_medium.csv`
- `test_noisy_high.csv`

In [ ]:
from google.colab import files
import os

DATA_DIR = '/content/data/final'
os.makedirs(DATA_DIR, exist_ok=True)

print('Select: train.csv, val.csv, test_noisy_medium.csv, test_noisy_high.csv')
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(f'{DATA_DIR}/{fname}', 'wb') as f:
        f.write(content)

print('\nUploaded:')
for f in sorted(os.listdir(DATA_DIR)):
    rows = sum(1 for _ in open(f'{DATA_DIR}/{f}')) - 1
    print(f'  {f}: {rows:,} rows')

## Step 3: Setup

In [ ]:
import os, json
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Only medium and high
NOISE_LEVELS = ['medium', 'high']

def load_data(noise_level):
    train_df = pd.read_csv(f'{DATA_DIR}/train.csv').dropna(subset=['text','label'])
    val_df   = pd.read_csv(f'{DATA_DIR}/val.csv').dropna(subset=['text','label'])
    # train on clean + this noise level only
    train_df = train_df[train_df['noise_level'].isin(['clean', noise_level])]
    test_df  = pd.read_csv(f'{DATA_DIR}/test_noisy_{noise_level}.csv').dropna(subset=['text','label'])
    print(f'  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
    return train_df, val_df, test_df

def save_result(result, name):
    out = f'{RESULTS_DIR}/{name}.json'
    with open(out, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'  Saved -> {out}')

print('Setup complete.')

## Step 4: CharCNN — medium and high

In [ ]:
CHARS = list(' abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,!?@#&\'- ') + ['<PAD>','<UNK>']
CHAR2IDX  = {c: i for i, c in enumerate(CHARS)}
PAD_IDX   = CHAR2IDX['<PAD>']
UNK_IDX   = CHAR2IDX['<UNK>']
VOCAB_SIZE = len(CHARS)

def text_to_ids(text, max_len=300):
    ids = [CHAR2IDX.get(c, UNK_IDX) for c in str(text)[:max_len]]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids

class CharDataset(Dataset):
    def __init__(self, texts, labels):
        self.data   = [torch.tensor(text_to_ids(t), dtype=torch.long) for t in texts]
        self.labels = labels
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        return self.data[i], torch.tensor(self.labels[i], dtype=torch.long)

class CharCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, 64, padding_idx=PAD_IDX)
        self.convs  = nn.ModuleList([nn.Conv1d(64, 128, fs) for fs in (3, 4, 5)])
        self.dropout = nn.Dropout(0.5)
        self.fc      = nn.Linear(128 * 3, 2)
    def forward(self, x):
        e = self.embedding(x).permute(0, 2, 1)
        pooled = [F.max_pool1d(F.relu(c(e)), c(e).size(2)).squeeze(2) for c in self.convs]
        return self.fc(self.dropout(torch.cat(pooled, dim=1)))

all_results = []
print('-- CharCNN: medium and high --')

for noise in NOISE_LEVELS:
    print(f'\nnoise={noise}')
    train_df, val_df, test_df = load_data(noise)

    train_loader = DataLoader(CharDataset(train_df['text'].tolist(), train_df['label'].tolist()),
                              batch_size=128, shuffle=True)
    val_loader   = DataLoader(CharDataset(val_df['text'].tolist(),   val_df['label'].tolist()),
                              batch_size=128)
    test_loader  = DataLoader(CharDataset(test_df['text'].tolist(),  test_df['label'].tolist()),
                              batch_size=128)

    model     = CharCNN().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    best_val_f1, best_state = 0, None

    for epoch in range(10):
        model.train()
        for X, y in tqdm(train_loader, desc=f'  Epoch {epoch+1}/10', leave=False):
            optimizer.zero_grad()
            criterion(model(X.to(DEVICE)), y.to(DEVICE)).backward()
            optimizer.step()

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for X, y in val_loader:
                preds.extend(model(X.to(DEVICE)).argmax(1).cpu().tolist())
                targets.extend(y.tolist())
        val_f1 = f1_score(targets, preds, average='macro')
        print(f'  Epoch {epoch+1:2d} | Val F1: {val_f1:.4f}')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X, y in test_loader:
            preds.extend(model(X.to(DEVICE)).argmax(1).cpu().tolist())
            targets.extend(y.tolist())
    test_f1 = f1_score(targets, preds, average='macro')
    print(f'Test F1: {test_f1:.4f}')
    print(classification_report(targets, preds, target_names=['Non-hate','Hate']))

    result = {
        'model': 'CharCNN', 'noise_level': noise,
        'best_val_f1': round(best_val_f1, 4), 'test_f1_macro': round(test_f1, 4),
        'epochs': 10, 'lr': 0.001
    }
    save_result(result, f'char_cnn_{noise}')
    all_results.append(result)

print('\nDone!')
for r in all_results:
    print(f"  {r['noise_level']:8s} | val F1: {r['best_val_f1']:.4f} | test F1: {r['test_f1_macro']:.4f}")

## Step 5: Download the 2 JSON files

In [ ]:
from google.colab import files

# Download both result files directly
for noise in ['medium', 'high']:
    path = f'{RESULTS_DIR}/char_cnn_{noise}.json'
    if os.path.exists(path):
        files.download(path)
        print(f'Downloading {path}')
    else:
        print(f'Not found: {path} -- check if training completed')

print('\nSend these 2 JSON files back:\n  char_cnn_medium.json\n  char_cnn_high.json')